In [ ]:
import gradio as gr
import os
import stripe
import pdfplumber
import uuid
from flask import Flask, request
from new_functions import (
    extract_resume_text,
    sanitize_input,
    prompt_creator,
    get_resume_response,
    process_resume,
    export_resume,
    cover_letter_prompt_creator,
    get_cover_response,
    save_cover_letter,
    is_posting_recent,
    template_detector,
    mentioned_on_socials,
    detect_urgency_language
) 

stripe.api_key = os.getenv("STRIPE_SECRET_KEY")
PRICE_ID = "price_1S4MQICB2P1PAV6iRNFAcF36"

# tracking credits / resumes
user_credits = {}
active_resumes = {}
FREE_CREDITS = 3

def get_user_id():
    """
    Gets user's id; if no login info provided, uses session-based ID.
    If needed, replaces with real authentication later.
    """
    return str(uuid.uuid4())

# create the permanent user id
user_id = get_user_id()

def run_resume_with_credits(resume_file, job_input):
    user_id = get_user_id()
    credits = user_credits.get(user_id, FREE_CREDITS)

    # generate the resume's unique ID from uploaded file and job description
    resume_name = getattr(resume_file, "name", "unknown")
    new_resume_id = f"{resume_name}_{hash(job_input)}"

    # run a check to see if it's a new resume (using a credit) or just an edit to the same one (not using any credits)
    if active_resumes.get(user_id) != new_resume_id:
        # make sure it's NOT an unlimited subscription
        if credits != float("inf"):
            if credits <= 0:
                return ("⏰ Looks like your 3 free resumes have been completed, but we'd love to keep helping - our prompts are constantly being worked on to improve your likelihood of landing your dream job. Please consider subscribing - at only $5.99 / month, you get all the AI-powered resume optimization you want!", "", "", f"Free resumes left: 0")
            user_credits[user_id] = credits - 1
            credits -= 1
        active_resumes[user_id] = new_resume_id

    # normal resume generation
    result = process_resume(resume_file, job_input)
    return (*result, f"Free resumes left: {'∞' if credits == float('inf') else credits}")

def create_checkout_session():
    try:
        session = stripe.checkout.Session.create(
            client_reference_id = user_id,
            payment_method_types = ['card'],
            line_items = [{
                'price' : PRICE_ID,
                'quantity' : 1,
            }],
            mode = 'subscription', 
            success_url = "https://www.resumewhip.com/success",
            cancel_url = "https://www.resumewhip.com/cancel"
        )

        return session.url
    except Exception as e:
        return f"There was an error creating your checkout session: {e}"

with gr.Blocks(title = "ResumeWhip") as app:

    gr.HTML("""<head>
            <link rel="icon" href="favicon.png" type="image/png">
            </head>""")
    # --- Header ---
    gr.Markdown("""
    <h1 style='text-align:center; color:#1e90ff;'>🏎️💨 Welcome To ResumeWhip!!</h1>
    <h2 style='text-align:center; color:#dd1eff;'>Your One-Stop AI-Powered Resume Optimizer Shop</h2> 
    <h3 style='text-align:center;'>Powerful Simplicity: Just Verify → Upload → Optimize → Apply!</h3>          
    """)

    with gr.Row():
        # --- Sidebar (simplified into Accordions) ---
        with gr.Column(scale=1):
            with gr.Accordion("🦮 How To Use This Website", open = False):
                gr.Markdown("""
                        1.) Crank Your Existing Resume Up To 11 - list every single skill and experience you have
                            (this is how our AI writes your resume and scores your chances);  
                        2.) Follow the Prompts To Load the Requested Info;  
                        3.) Choose Your Tool (you don't have to use all 3);  
                        4.) Proofread / Edit the Results Using the 'Copy/Pastes' below;  
                        5.) Download Your File as PDF; and  
                        6.) Apply!
                            """)
                
            with gr.Accordion("📚 Copy/Pastes for Resume Formatting", open=False):
                gr.Code("""
If you're not happy with the default resume 
format, you can make adjustments using the 
simple copy and pastes below:
                        
Want To Change Fonts:
# = Biggest  
## = Smaller  
### = Smallest
<b>text</b> = Bold  
<i>text</i> = Italic  
<u>text</u> = Underline
(⬆️ Can Be Combined)
                        
How About A List?
- = Bullet Point  
1. = Numbered List

Link Your Website:
[Your Website](https://www.yourwebsite.com)

Too "clumpy?" Break things up into separate lines
with two spaces where you want the line break
(e.g. after a period). 
                        
Or, if you want to break things up by drawing a
a line, just enter three dashes (---) where you 
want the section break (eg: between 'Summary' 
and 'Experience).

Page cuts off where you don't want it to?
Force Create A New Page by copy/pasting this entire 
line, andnput it wherever you want:
<div style="page-break-after: always; break-after: page;"></div>                      
                """, language="markdown")
    # Stripe Subscribe Button
            gr.HTML("""
        <div style="text-align:center; margin-top:20px;">
            <a href="https://buy.stripe.com/cNi9ASgWl6C614l3Ja1Jm00" 
               target="_blank" 
               style="background-color:#635BFF; color:white; padding:15px 25px; 
                      text-decoration:none; border-radius:8px; font-size:1.2em; 
                      font-weight:bold; display:inline-block;">
                🚀 Subscribe and Get Unlimited Resume Optimizatons
                    for Only $5.99/month!
            </a>
        </div>
    """)
            
            gr.Markdown("### 🛡️ We never store, share, or sell your data.")
            
            gr.Markdown("### 🔐 All payments are handled through Stripe.")

            gr.Markdown("[📬 Need Help / Have Suggestions? Send Us An Email!](mailto:support@resumewhip.com)")

            gr.Markdown("### #️⃣ Know someone who could use this in their job search? Share away!")
            gr.HTML("""
                    <!-- Share Icons -->
    <div style="display:flex; flex-direction:row; gap:15px; justify-content:center; margin-top:10px;">
        <a href="https://www.facebook.com/sharer/sharer.php?u=https://resumewhip.com" target="_blank">
            <img src="https://upload.wikimedia.org/wikipedia/commons/0/05/Facebook_Logo_%282019%29.png" style="width:32px; height:32px;">
        </a>
        <a href="https://x.com/intent/post?url=https://resumewhip.com&text=Check%20out%20this%20awesome%20Resume%20Optimizer!" target="_blank">
            <img src="https://upload.wikimedia.org/wikipedia/commons/c/ce/X_logo_2023.svg" alt="X" style="width:36px; height:36px;">
        </a>
        <a href="https://www.linkedin.com/sharing/share-offsite/?url=https://resumewhip.com" target="_blank">
            <img src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" style="width:32px; height:32px;">
        </a>
        <a href="https://www.reddit.com/submit?url=https://resumewhip.com&title=Check%20this%20out!" target="_blank">
             <img src="https://cdn.simpleicons.org/reddit/FF4500" alt="Reddit" style="width:32px; height:32px;">
        </a>
    </div>
""")
#             gr.Markdown("### 💸 Donations appreciated... only if we've helped:")
#             gr.HTML("""
#                     <div style="text-align:left; display:flex; flex-direction:column; gap:10px; margin-top:15px;">
#         <!-- Support Buttons -->
#         <form action="https://www.paypal.com/donate" method="post" target="_blank">
#             <input type="hidden" name="business" value="YOUR_PAYPAL_EMAIL_OR_ID" />
#             <input type="hidden" name="currency_code" value="USD" />
#             <input type="image" src="https://www.paypalobjects.com/en_US/i/btn/btn_donate_LG.gif" 
#                border="0" name="submit" alt="Donate with PayPal" />
#         </form>
#         <a href="https://www.buymeacoffee.com/yourname" target="_blank">
#         <img src="https://cdn.buymeacoffee.com/buttons/default-orange.png" style="height:40px;width:180px;">
#         </a>
#     </div>
# </div>
# """)

        # --- Main App ---      
        # Add CSS styling for the counter
        gr.HTML("""
                <style>
                #resume-counter {
                text-align: center;
                font-size: 1.2em;
                color: #1e90ff;
                margin-bottom: 10px;
                font-weight: bold;
                }
                </style>
                """)
        
        with gr.Column(scale=5):
            with gr.Row():
                resume_input = gr.File(label="📝 Upload Your Resume Here", file_types = [".pdf", ".md", ".docx", ".txt"])
                company_input = gr.Textbox(label="🏢 Type In the Company Name", placeholder="e.g., ING Partners")
                job_input = gr.Textbox(label="🔬 Paste Entire Job Description", lines=8)
                
            with gr.Row():
                resume_counter = gr.Markdown("### Free Resumes Left: 3", elem_id = "counter")

            gr.Markdown("<h2 style='text-align:center; color:#ff7f50;'>🧰 Tools In the Toolkit</h2>")

            with gr.Tab("📋 Job Validator"):
                with gr.Row():
                    # with gr.Column():
                    #     resume_input = gr.File(label="Upload Your Resume")
                    #     job_desc_input = gr.Textbox(label="Paste Job Description", lines=10)
                        # resume_input = gr.File(label="Upload Your Resume")
                        # job_desc_input = gr.Textbox(label="Paste Job Description", lines=10)
                    # with gr.Column():
                        jd_date = gr.Textbox(label="Posting Date (YYYY-MM-DD)")
                        jd_title = gr.Textbox(label="Job Title")

                validate_btn = gr.Button("✅ Validate Job - A Quick Check To See If the Job Post Is Legitimate")
                validation_output = gr.Markdown(label="Job Validation Results")

                def full_job_validator(job_input_text, posting_date, company, job_title):
                    # --- Warning for empty / missing company name ---
                    if not company or company.strip() == "":
                        return "⚠️ Please enter a company name so we can validate the job posting."
                    
                    # --- Warning if job_input_text is missing ---
                    if not job_input_text or job_input_text.strip() == "":
                        return "⚠️ Sorry, but we need a job description to help validate the job!"
                    
                    # --- Warning if posting_date is missing ---
                    if not posting_date or posting_date.strip() == "":
                        return "⚠️ Can you please give us a job posting date (or your best guess)? That would help alot."
                    
                    # --- Existing job validation logic ---
                    recent = is_posting_recent(posting_date)
                    template_flag = template_detector(job_input_text)
                    urgency_flag = detect_urgency_language(job_input_text)
                    social_links = mentioned_on_socials(company, job_title)

                    report = "### 🕒 Posting Date Check:\n"
                    report += "✅ Job appears to be recent enough.\nNot a lot of jobs are still looking after 45 days." if recent else "⚠️ Warning! Job may be outdated.\n"

                    report += "\n### 🤖 Template Language:\n"
                    report += "⚠️ Generic/template language detected - could be just harvesting data and / or candidates.\n" if template_flag else "✅ Posting looks specific enough to be an actual need.\n"

                    report += "\n### ⚡ Urgency Language:\n"
                    report += "⚠️ Urgency words detected.\nProceed with caution, especially if the post is older than 30 days." if urgency_flag else "✅ No urgency language detected.\n"

                    report += "\n### 🔍 Social Media Mentions:\n"
                    report += f"- [Search on X](<{social_links['x']}>)\n"
                    report += f"- [Search on LinkedIn](<{social_links['linkedin']}>)\n"

                    # # --- Optional: Resume processing ---
                    # if resume_file is not None:
                    #     resume_report = process_resume(resume_file, job_input)
                    #     report += f"\n### 📄 Resume Fit Analysis:\n{resume_report}\n"

                    return report

                validate_btn.click(
                    full_job_validator,
                    inputs=[job_input, jd_date, company_input, jd_title],
                    outputs=validation_output
                )

            # with gr.Tab("Job Validator"):
            #     jd_date = gr.Textbox(label="Posting Date (YYYY-MM-DD)")
            #     jd_title = gr.Textbox(label="Job Title")
            #     jd_validate_btn = gr.Button("✅ Validate Job")
            #     jd_validation_result = gr.Markdown()

            #     def validate_job(posting_date, company, job_title, job_description):
            #         recent = is_posting_recent(posting_date)
            #         template_flag = template_detector(job_description)
            #         urgency_flag = detect_urgency_language(job_description)
            #         social_links = mentioned_on_socials(company, job_title)

            #         report = f"### 🕒 Posting Date Check:\n"
            #         report += "✅ Job appears to be recent.\n" if recent else "⚠️ Job may be outdated.\n"

            #         report += f"\n### 🤖 Template Language:\n"
            #         report += "⚠️ Generic/template language detected - could be just harvesting candidates.\n" if template_flag else "✅ Posting looks specific enough to be an actual need.\n"

            #         report += f"\n### 🔍 Social Media Mentions:\n"
            #         report += f"- [Search on X](<{social_links['x']}>)\n"
            #         report += f"- [Search on LinkedIn](<{social_links['linkedin']}>)\n"

            #         return report

            #     jd_validate_btn.click(
            #         fn=validate_job,
            #         inputs=[jd_date, company_input, jd_title, job_input],
            #         outputs=jd_validation_result
            #     )
            
            with gr.Tab("Resume Optimizer"):
                run_resume = gr.Button("🧙 Click Here To Whip Up Some Resume Magic!")
                resume_md = gr.Markdown()
                resume_edit = gr.Textbox(label="Above Is the Preview of Your Optimized Resume - If You Want, You Can Edit In This Box Below.", lines=10)
                suggestions = gr.Markdown(label="Suggestions")
                export_resume_btn = gr.Button("⬇ Download as PDF")
                export_resume_result = gr.File()

            with gr.Tab("Cover Letter Generator"):
                run_cover = gr.Button("📝 Click Here To Whip Up A Cover Letter")
                cover_output = gr.Textbox(label="Cover Letter", lines=12)
                export_cover_btn = gr.Button("⬇ Download as PDF")
                export_cover_result = gr.File()
            
            # buy_button = gr.Button("🛍️ Subscribe and Get All the Resumes You Want for Less Than $6/Month!")
            # buy_link = gr.Markdown()

            # buy_button.click(
            #     fn = lambda: f"[Click here for Limitless AI-Powered Resume Optimization!]({create_checkout_session()})",
            #     outputs = buy_link
            # )

            # with gr.Tab("Job Validator"):
            #     jd_date = gr.Textbox(label="Posting Date (YYYY-MM-DD)")
            #     jd_title = gr.Textbox(label="Job Title")
            #     jd_validate_btn = gr.Button("✅ Validate Job")
            #     jd_validation_result = gr.Markdown()

            #     def validate_job(posting_date, company, job_title, job_description):
            #         recent = is_posting_recent(posting_date)
            #         template_flag = template_detector(job_description)
            #         social_links = mentioned_on_socials(company, job_title)

            #         report = f"### 🕒 Posting Date Check:\n"
            #         report += "✅ Job appears to be recent.\n" if recent else "⚠️ Job may be outdated.\n"

            #         report += f"\n### 🤖 Template Language:\n"
            #         report += "⚠️ Generic/template language detected.\n" if template_flag else "✅ Looks specific.\n"

            #         report += f"\n### 🔍 Social Media Mentions:\n"
            #         report += f"- [Search on X](<{social_links['x']}>)\n"
            #         report += f"- [Search on LinkedIn](<{social_links['linkedin']}>)\n"

            #         return report

            #     jd_validate_btn.click(
            #         fn=validate_job,
            #         inputs=[jd_date, company_input, jd_title, job_input],
            #         outputs=jd_validation_result
            #     )

            # Resume events
            run_resume.click(fn=run_resume_with_credits, inputs=[resume_input, job_input], outputs=[resume_md, resume_edit, suggestions, resume_counter])
            export_resume_btn.click(fn=export_resume, inputs=[resume_edit, company_input], outputs=[export_resume_result])

            # Cover letter events
            def generate_cover_letter(resume_file, job_input):
                if resume_file is None or not job_input or job_input.strip() == "":
                    return "⚠️ Sorry, but the tools need both a resume and pasted job description before they can do you any good."

                # Normalize extension safely
                fname = getattr(resume_file, "name", "")
                ext = fname.lower().split(".")[-1] if "." in fname else ""

            # Read resume text from PDF or text-like files
                resume_txt = ""
                try:
                    if ext == "pdf":
                        with pdfplumber.open(fname or resume_file) as pdf:
                            for page in pdf.pages:
                                resume_txt += (page.extract_text() or "")
                    else:
                    # Fallback: treat as text/markdown
                        with open(fname, "r", encoding="utf-8") as f:
                            resume_txt = f.read()
                except Exception as e:
                    return f"😐 Ruh-roh! Problem with the resume: {e}"

                prompt = cover_letter_prompt_creator(resume_txt, job_input)
                return get_cover_response(prompt)


            run_cover.click(fn=generate_cover_letter, inputs=[resume_input, job_input], outputs=[cover_output])
            export_cover_btn.click(fn=save_cover_letter, inputs=[cover_output, company_input], outputs=[export_cover_result])

    # --- Footer ---
    # gr.Markdown("""
    # <hr>
    # <p style='text-align:center; font-size:1.5em; color:gray;'>
    # 🛡️ Your data is never stored, shared, or sold. Ever.
    # </p>
    # """)

# set up Flask so we can set up the webhook
flask_app = Flask(__name__)

# Webhook so that once users pay, they have full access

@flask_app.route("/webhook", methods = ["POST"])

def stripe_webhook():
    payload = request.data
    signature_header = request.headers.get("Stripe-Signature")
    endpoint_secret = os.getenv("STRIPE_WEBHOOK_SECRET")

    try:
        event = stripe.Webhook.construct_event(payload, signature_header, endpoint_secret)
    except Exception as e:
        return str(e), 400
    
    if event["type"] == "checkout.session.completed":
        session = event["data"]["object"]
        user_id = session.get("client_reference_id", "guest")

        # give user infinite ("inf") access
        user_credits[user_id] = float("inf")

    return "", 200

# Launch
if __name__ == "__main__":
    app.launch(server_name="0.0.0.0", server_port=int(os.environ.get("PORT", "8080")))
    flask_app.run(host = "0.0.0.0", port = 8081)

2025-09-06 22:42:53,653 - INFO - HTTP Request: GET http://localhost:8080/gradio_api/startup-events "HTTP/1.1 200 OK"
2025-09-06 22:42:53,663 - INFO - HTTP Request: HEAD http://localhost:8080/ "HTTP/1.1 200 OK"


* Running on local URL:  http://0.0.0.0:8080
* To create a public link, set `share=True` in `launch()`.


 * Serving Flask app '__main__'
 * Debug mode: off


2025-09-06 22:42:54,259 - INFO - WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8081
 * Running on http://192.168.1.101:8081
2025-09-06 22:42:54,261 - INFO - Press CTRL+C to quit
